# Knowledge graph pipeline

Runs the `semantic_corpus` command-line pipeline from a query to an interactive knowledge graph (KG):

```
pipeline --workdir <folder> --query "AMOC" --limit 30 --target clusters --kg
```

**Steps:** `collect` (download papers from Europe PMC) → `corpus` → `phrases` → `clusters` → `kg`.

**How to use:** run the cells in order (*Runtime → Run all*). Change the query in section 2.

## 1. Install

Gets the code from the `pmr202610` branch of `semantic_corpus` and the `pmr202601` branch of `encyclopedia`. `docling` and `torch` are skipped on purpose: they are large and this pipeline does not use them. Takes about 1 minute.

In [ ]:
import subprocess
import sys
from pathlib import Path

BRANCH = "pmr202610"
ENCYCLOPEDIA_BRANCH = "pmr202601"  # has encyclopedia.pipeline; main does not
ROOT = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO = ROOT / "semantic_corpus"
ENCYCLOPEDIA_REPO = ROOT / "encyclopedia"
OUTPUT_ROOT = ROOT / "kg_outputs"


def run(*command):
    subprocess.run([str(part) for part in command], check=True)


def get_code(path, branch, only=None):
    if path.exists():
        run("git", "-C", path, "fetch", "-q", "--depth", "1", "origin", branch)
        run("git", "-C", path, "checkout", "-q", "FETCH_HEAD")
        return
    partial = ["--filter=blob:none", "--sparse"] if only else []
    run("git", "clone", "-q", "--depth", "1", *partial, "--branch", branch,
        f"https://github.com/semanticClimate/{path.name}.git", path)
    if only:
        run("git", "-C", path, "sparse-checkout", "set", "--no-cone", *only)


get_code(REPO, BRANCH)
# The encyclopedia repository is large; only its Python code is needed.
get_code(ENCYCLOPEDIA_REPO, ENCYCLOPEDIA_BRANCH,
         only=["/encyclopedia/", "/Examples/*.py"])

run(sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO)
run(sys.executable, "-m", "pip", "install", "-q",
    "bagit", "requests", "beautifulsoup4", "lxml", "tqdm", "coloredlogs",
    "configargparse", "pyyaml", "doit", "yake", "scikit-learn", "networkx",
    "pandas", "amilib")
print("Install complete.")

## 2. Choose the query

- Use `AND`, `OR` and `NOT` in capitals, for example `AMOC AND insolation`.
- `LIMIT` is the number of papers. Allow roughly 10–15 seconds per paper for the download.
- Turn on `ADD_ENCYCLOPEDIA` to also look terms up in Wikipedia. `ENCYCLOPEDIA_ENTRIES` sets how many terms; each one takes several seconds, so 100 terms can add more than 20 minutes.

Each query and limit gets its own folder, so changing either one starts a new run. Repeating the same values reuses the results.

In [ ]:
import re

QUERY = "AMOC"  #@param {type:"string"}
LIMIT = 30  #@param {type:"slider", min:5, max:100, step:5}
ADD_ENCYCLOPEDIA = False  #@param {type:"boolean"}
ENCYCLOPEDIA_ENTRIES = 30  #@param {type:"slider", min:10, max:100, step:10}

slug = re.sub(r"[^a-z0-9]+", "_", QUERY.lower()).strip("_")[:50] or "query"
WORKDIR = OUTPUT_ROOT / f"{slug}_{LIMIT}"
if ADD_ENCYCLOPEDIA:
    WORKDIR = WORKDIR.with_name(f"{WORKDIR.name}_encyclopedia_{ENCYCLOPEDIA_ENTRIES}")
print("Results folder:", WORKDIR)

## 3. Run the pipeline

The title of each paper is printed as it is downloaded. If the titles show that the query is wrong, stop the cell (■ button): the pipeline stops too. The cell raises an error if the pipeline fails.

In [ ]:
import os
import time

target = "encyclopedia" if ADD_ENCYCLOPEDIA else "clusters"
command = [
    sys.executable, "-c",
    "import sys; from semantic_corpus.pipeline.cli import main; sys.exit(main())",
    "--workdir", str(WORKDIR), "--query", QUERY, "--limit", str(LIMIT),
    "--target", target, "--kg", "--no-open",
]
if ADD_ENCYCLOPEDIA:
    command += ["--max-entries", str(ENCYCLOPEDIA_ENTRIES)]
environment = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONUTF8": "1",
               "PYTHONPATH": str(ENCYCLOPEDIA_REPO)}

started = time.time()
process = subprocess.Popen(command, cwd=REPO, env=environment, text=True, encoding="utf-8",
                           errors="replace", stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
try:
    for line in process.stdout:
        print(line, end="")
    process.wait()
except KeyboardInterrupt:
    # Stopping the cell would otherwise leave the pipeline running in the background.
    process.terminate()
    process.wait()
    print("\nStopped. Change the query in section 2 and run this cell again.")
    raise
if process.returncode != 0:
    raise RuntimeError(f"pipeline failed with exit code {process.returncode}")
print(f"Finished in {(time.time() - started) / 60:.1f} minutes.")

## 4. Results

In [ ]:
import json

import pandas as pd
from IPython.display import display

clusters_data = json.loads((WORKDIR / "clusters" / "clusters.json").read_text(encoding="utf-8"))
cluster_summary = json.loads((WORKDIR / "clusters" / "graph_summary.json").read_text(encoding="utf-8"))
kg_summary = json.loads((WORKDIR / "kg" / "kg_summary.json").read_text(encoding="utf-8"))

print(f"Papers:              {cluster_summary['paper_count']}")
print(f"Clusters:            {cluster_summary['cluster_count']}")
print(f"Papers in no cluster: {cluster_summary['orphan_paper_count']}")
print(f"KG nodes / edges:    {kg_summary['counts']['nodes']} / {kg_summary['counts']['edges']}")

display(pd.DataFrame(
    {
        "cluster": cluster.get("cluster_id"),
        "theme": cluster.get("theme"),
        "papers": len(cluster.get("paper_ids", [])),
    }
    for cluster in clusters_data["clusters"]
))

## 5. Knowledge graph

Drag nodes, zoom, and hover for details. Node positions change each time the graph is drawn; the data does not.

Use the **Louvain resolution** slider to re-cluster: lower values give fewer, larger clusters. The slider needs the small server this cell starts, so it only works while the Colab session is running.

In [ ]:
# The package was installed during this session; add it to the path so no restart is needed.
sys.path.insert(0, str(REPO))
from semantic_corpus.cluster_review.kg_server import serve_kg

if "kg_server" in globals():  # running the cell again: stop the previous server
    kg_server.shutdown()
    kg_server.server_close()
kg_server = serve_kg(WORKDIR, port=8765, open_browser=False, block=False)
kg_port = kg_server.server_address[1]

try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(kg_port, path="/knowledge_graph.html", height=720)
except ImportError:
    print(f"Open http://127.0.0.1:{kg_port}/knowledge_graph.html in a browser.")

## 6. Download the results

Colab files are deleted when the session ends. This saves the whole results folder as a `.zip`, including `kg/knowledge_graph.html`.

In [ ]:
import shutil

archive = shutil.make_archive(str(WORKDIR), "zip", WORKDIR)
print("Created:", archive)

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass